# Image-to-Video Generation with stable-diffusion.cpp

This notebook demonstrates how to generate videos from images using stable-diffusion.cpp.

## What This Notebook Does

1. Loads a pre-trained image-to-video model
2. Takes your input image
3. Applies motion based on your text prompt
4. Saves the output as a video file

## Requirements

- Python 3.8+
- stable-diffusion.cpp
- ffmpeg (for video encoding)
- PIL/Pillow for image processing

## Hardware Requirements

- **Minimum**: 16 GB RAM
- **Recommended**: 24 GB+ RAM or 10 GB+ VRAM
- **Optimal**: 32 GB+ RAM or 16 GB+ VRAM

## Model Selection Guide

| RAM/VRAM | Model | Quality | Speed |
|----------|-------|---------|-------|
| 16 GB+ | stabilityai/stable-video-diffusion-img2vid | Medium | Medium |
| 24 GB+ | stabilityai/stable-video-diffusion-img2vid-xt | High | Slow |

## How It Works (Simple Explanation)

1. **Image Encoder**: Extracts features from your input image
2. **Text Encoder**: Converts your text prompt into numbers (embeddings)
3. **Noise Scheduler**: Starts with noise around the image
4. **U-Net Model**: Repeatedly refines the image, guided by your text
5. **Frame Generation**: Creates each video frame step-by-step
6. **Video Assembly**: Combines frames into a video file

## Getting Help

If you see errors:
1. Check you you have enough RAM/VRAM
2. Try a smaller model
3. Reduce the video length or resolution
4. Check the documentation in `docs/` folder

In [ ]:
# Step 1: Import required libraries
import sys
import os
from PIL import Image
import numpy as np

# Add scripts directory to path
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'scripts'))

print("Libraries imported successfully")

In [ ]:
# Step 2: Load the model

def load_model(model_name="stabilityai/stable-video-diffusion-img2vid"):
    """Load an image-to-video model""
    
    print(f"Loading model: {model_name}")
    print("Please wait...")
    
    try:
        # Import stable-diffusion.cpp
        import sd2cpp
        
        # Load the pipeline
        pipe = sd2cpp.ImageToVideoPipeline(model_name=model_name)
        print(f"Model loaded successfully!")
        return pipe
    except Exception as e:
        print(f"Error loading model: {e}")
        print("\nCommon issues:")
        print("1. Not enough GPU memory - try smaller model or CPU")
        print("2. Internet connection needed for first download")
        print("3. Install stable-diffusion.cpp: bash scripts/install_local_video.sh")
        raise

# Load model
model_name = "stabilityai/stable-video-diffusion-img2vid"
pipe = load_model(model_name)

In [ ]:
# Step 3: Generate video from image

def generate_video_from_image(pipe, image, prompt, num_frames=8):
    """Generate a video from an image and text prompt""
    
    print(f"\nGenerating video from image...")
    print(f"Prompt: {prompt}")
    print(f"Frames: {num_frames}")
    print("This may take a few minutes...")
    
    try:
        # Convert PIL image to numpy array if needed
        if isinstance(image, Image.Image):
            image_array = np.array(image)
        else:
            image_array = image
        
        # Generate the video frames
        frames = pipe.generate(
            image_array,
            prompt,
            num_frames=num_frames,
            num_inference_steps=30,
        )
        
        print(f"\nGenerated {len(frames)} frames!")
        return frames
        
    except Exception as e:
        print(f"\nError during generation: {e}")
        print("\nTry:")
        print("1. Reduce num_frames (try 4 instead 8)")
        print("2. Use a smaller model")
        raise

# Load an image (replace with your image path)
image_path = None  # Set this to your image path, e.g., "/path/to/image.jpg"

if image_path and os.path.exists(image_path):
    image = Image.open(image_path)
    print(f"Loaded image: {image.size}")
else:
    print("No image loaded. Please set image_path to your image file.")
    print("Example: image_path = '/Users/seanbailey/Desktop/test.jpg'")
    # Create a placeholder image for demonstration
    image = Image.new('RGB', (256, 256), color='red')
    print("Using placeholder red image for demonstration")

# Your text prompt goes here
prompt = "A peaceful animated scene with gentle motion"

# Generate video
frames = generate_video_from_image(
    pipe,
    image,
    prompt,
    num_frames=8    # Number of frames (4=short, 8=medium, 16=longer)
)

In [ ]:
# Step 4: Save the generated video

import cv2

def save_video(frames, prompt, output_dir="output"):
    """Save generated frames as a video file""
    
    os.makedirs(output_dir, exist_ok=True)
    
    safe_prompt = ''.join(c if c.isalnum() or c in ' -_' else '_' for c in prompt[:50])
    output_path = os.path.join(output_dir, f"video_from_image_{safe_prompt}.mp4")
    
    if not frames:
        print("No frames to save!")
        return None
    
    # Get frame dimensions
    first_frame = frames[0]
    if hasattr(first_frame, 'size'):
        height, width = first_frame.size[1], first_frame.size[0]
    else:
        height, width = first_frame.shape[:2]
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc('mp4v')
    video = cv2.VideoWriter(output_path, fourcc, 4.0, (width, height))
    
    # Write each frame
    for i, frame in enumerate(frames):
        if hasattr(frame, 'convert'):
            frame = frame.convert('RGB')
        
        if hasattr(frame, 'numpy'):
            frame_np = frame.numpy()
        else:
            frame_np = np.array(frame)
        
        frame_bgr = cv2.cvtColor(frame_np, cv2.COLOR_RGB2BGR)
        video.write(frame_bgr)
    
    video.release()
    print(f"\nVideo saved: {output_path}")
    return output_path

# Save the generated video
video_path = save_video(frames, prompt)
print(f"\nYour video is ready at: {video_path}")

## Next Steps

1. **Watch your video**: Open the saved file in your video player
2. **Try different images**: Upload different images
3. **Try different prompts**: Change the text to describe different motion
4. **Adjust quality**: Change `num_frames` for different results

## Troubleshooting

### Out Memory Error
If you see "Out Memory" or "CUDA out memory":
- Reduce `num_frames` to 4
- Use a smaller model
- Close other applications to free RAM

### Slow Generation
If generation is too slow:
- Use fewer frames (4 instead 8)
- Enable GPU if available

### Missing Packages
If you see import errors:
```bash
pip install opencv-python pillow
```